# AB-200 Questions: Q11–20 — Python Syntax and Fundamentals

Part of a series — each notebook covers 10 questions from the AB-200 list. See [Q1–10](Q-1-10-solutions.ipynb) for the first set.

For each question we follow the same drill, in this order:

1. **Original question** — quoted as written, with its hint, so the notebook stands on its own.
2. **Restate the problem** — in plain language, so we're sure we're solving the right thing.
3. **Think algorithmically** — write the steps as an ordinary person would describe them, with no loops/ifs/syntax.
4. **Brute force** — the first correct idea that comes to mind, even if it's wasteful.
5. **Optimal** — built *on top of* the brute force by asking "what work are we repeating, and can we avoid it?"
6. **Tests** — normal case, edge cases, and asserts that actually run.
7. **Interview traps** — the specific mistakes that make candidates lose points on this exact question.

---
## Q11. Remove Vowels from String

> **Remove Vowels from String:** Remove all vowels from a given string.
> _Hint:_ Use a filter or list comprehension to exclude characters in the set of vowels.

**Restate:** Given a string, produce a new string containing every character *except* the vowels (`a, e, i, o, u`), preserving the order of everything that's kept. As with Q9, we'll treat this case-insensitively (remove both `'a'` and `'A'`) since real sentences have capitals and the question doesn't say otherwise.

**Algorithmic thinking (no syntax):**
1. Start a new, empty sequence.
2. Walk the original string one character at a time.
3. If the character is a vowel, skip it. Otherwise, place it onto the end of the new sequence.
4. Once you've walked the whole string, the new sequence is the answer.

This is the mirror image of Q9 (count the vowels) and Q2 (reverse a string) — same "walk once, decide per character" shape, but here the decision is "keep or drop" instead of "count" or "reposition."

In [ ]:
# Brute force — deliberately repeats Q2's O(n^2) trap: building a string
# with += inside a loop copies everything accumulated so far, every time.

def remove_vowels_brute(s):
    vowels = ['a', 'e', 'i', 'o', 'u', 'A', 'E', 'I', 'O', 'U']
    result = ""
    for char in s:
        if char not in vowels:
            result += char
    return result

print(remove_vowels_brute("Hello World"))   # "Hll Wrld"

**Optimal:** Collect the kept characters in a list comprehension (Python builds this in one pass with no repeated copying) and join once at the end, checking membership against a `set` rather than a `list`. This is O(n) time instead of O(n²).

In [ ]:
# Optimal — list comprehension + set membership + single join. O(n) time.

VOWELS_BOTH_CASE = frozenset('aeiouAEIOU')

def remove_vowels_optimal(s):
    return "".join(char for char in s if char not in VOWELS_BOTH_CASE)

print(remove_vowels_optimal("Hello World"))   # "Hll Wrld"

In [ ]:
# Tests

assert remove_vowels_optimal("") == ""                       # edge: empty string
assert remove_vowels_optimal("bcdfg") == "bcdfg"              # edge: no vowels to remove
assert remove_vowels_optimal("aeiouAEIOU") == ""              # edge: all vowels, both cases
assert remove_vowels_optimal("Hello World") == "Hll Wrld"
assert remove_vowels_optimal("y") == "y"                      # 'y' is not treated as a vowel

assert remove_vowels_brute("Hello World") == remove_vowels_optimal("Hello World")

print("All Q11 tests passed.")

**Interview traps:**
- **Same string-concatenation-in-a-loop trap as Q2.** If you find yourself writing `result += char` inside a loop, say out loud that you know it's O(n²) on `str` and that you'd switch to a list/generator + one `join()` for the real answer — that's the signal an interviewer is listening for, not just a working answer.
- **Case handling, again.** Removing only lowercase vowels and leaving `'A', 'E', 'I', 'O', 'U'` untouched is the single most common miss — decide and state your handling of case before coding, exactly as in Q9.
- **`filter()` is a legitimate third option**: `"".join(filter(lambda c: c not in VOWELS_BOTH_CASE, s))`. Functionally identical to the generator-expression version; mention it if asked for alternatives, but the generator expression is more readable and just as fast, so lead with that.
- **Don't confuse "remove vowels" with "remove `'y'` too."** As in Q9, unless explicitly asked, treat `'y'` as a consonant.

---
## Q12. Remove Duplicates from List

> **Remove Duplicates from List:** Given a list, remove duplicate elements while preserving the original order.
> _Hint:_ Use a set to track seen elements and build a new list with only the first occurrence of each element.

**Restate:** Given a list, produce a new list containing each distinct element exactly once, in the order it *first* appeared in the original.

**Algorithmic thinking (no syntax):**
1. Start a new, empty list, and a separate "memory" of everything you've already placed into it.
2. Walk the original list one element at a time.
3. If you've already seen this element before (it's in your memory), skip it.
4. Otherwise, place it into the new list *and* add it to your memory.
5. Once you've walked the whole list, the new list is the answer.

The phrase "preserving order" is doing real work in the restatement — it's the reason you can't just do `list(set(original))`. Sets have no guaranteed order, so converting to a set and back throws away the one thing the question explicitly asks you to keep.

In [ ]:
# Brute force — "memory" is a plain list, so checking "have I seen this?"
# is itself an O(n) linear scan. O(n^2) overall for n elements.

def remove_duplicates_brute(items):
    seen = []
    result = []
    for item in items:
        if item not in seen:
            result.append(item)
            seen.append(item)
    return result

print(remove_duplicates_brute([1, 2, 2, 3, 1, 4]))   # [1, 2, 3, 4]

**Optimal:** Swap the "memory" from a `list` to a `set` — the membership check drops from O(n) to O(1) average, taking the whole thing from O(n²) to O(n). This requires every element to be **hashable** (which is what makes `set` membership O(1) in the first place); we'll come back to what happens when it isn't, in the traps below.

In [ ]:
# Optimal — set for O(1) membership. O(n) time, O(n) space.

def remove_duplicates_optimal(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            result.append(item)
            seen.add(item)
    return result

print(remove_duplicates_optimal([1, 2, 2, 3, 1, 4]))   # [1, 2, 3, 4]

# Equivalent one-liner: relies on dict preserving insertion order (3.7+),
# and on dict keys being deduplicated automatically.
def remove_duplicates_dict_trick(items):
    return list(dict.fromkeys(items))

print(remove_duplicates_dict_trick([1, 2, 2, 3, 1, 4]))   # [1, 2, 3, 4]

In [ ]:
# Tests

assert remove_duplicates_optimal([]) == []                              # edge: empty list
assert remove_duplicates_optimal([1]) == [1]                             # edge: single element
assert remove_duplicates_optimal([1, 1, 1]) == [1]                       # edge: all duplicates
assert remove_duplicates_optimal([1, 2, 3]) == [1, 2, 3]                 # edge: no duplicates
assert remove_duplicates_optimal([3, 1, 2, 1, 3]) == [3, 1, 2]           # order = first occurrence
assert remove_duplicates_optimal(["a", "b", "a"]) == ["a", "b"]          # works for strings too

for items in ([], [1], [1, 1, 1], [3, 1, 2, 1, 3], [5, 4, 3, 2, 1]):
    assert remove_duplicates_brute(items) == remove_duplicates_optimal(items) == remove_duplicates_dict_trick(items)

print("All Q12 tests passed.")

**Interview traps:**
- **`list(set(items))` is the "obvious" wrong answer.** It removes duplicates correctly but does not preserve order — sets are unordered, so the resulting order is an implementation detail, not a guarantee. If "preserving order" is in the prompt (it is here), this answer fails the spec even though it "looks" right on small examples.
- **Unhashable elements break the set-based approach entirely.** A list of lists (`[[1, 2], [1, 2], [3]]`) can't go into a `set` — `TypeError: unhashable type: 'list'`. If the elements aren't guaranteed hashable, you have to fall back to the brute-force O(n²) approach (or a different structure entirely), and you should say so rather than assuming `set` always works.
- **`dict.fromkeys(items)` is a real, idiomatic one-liner**, not just a trick — it relies on two dict guarantees (insertion-order preservation since 3.7, and automatic key deduplication) that are worth being able to name explicitly if asked "why does this work?"
- **In-place vs. new list.** The question says "given a list, remove duplicates" — decide whether that means returning a *new* list (what we did) or mutating the original list in place. Both are defensible; state which one you're doing, since silently choosing one when the caller expected the other is a real bug in production code.

---
## Q13. Second Largest Number

> **Second Largest Number:** Find the second largest number in a list of integers.
> _Hint:_ Track the largest and second largest values in one pass (updating accordingly), or sort the list and pick the second last unique value.

**Restate:** Given a list of integers, find the second-largest **distinct** value — i.e. the largest value that is strictly less than the overall maximum. (This is the standard interpretation: in `[5, 5, 3]`, the second largest is `3`, not another `5` — otherwise "second largest" would just mean "the value at index 1 after sorting," which isn't a very interesting question. State this assumption explicitly, since the prompt doesn't spell it out.)

**Algorithmic thinking (no syntax):**
1. Keep track of two running numbers: the largest seen so far, and the second-largest seen so far.
2. Look at each number in the list, one at a time.
3. If it's bigger than the current largest, the old largest becomes the new second-largest, and this number becomes the new largest.
4. Otherwise, if it's bigger than the current second-largest *and* different from the current largest, it becomes the new second-largest.
5. After looking at every number, whatever is being tracked as "second-largest" is the answer.

Step 4's "and different from the current largest" is the detail that enforces "distinct values" — without it, a list like `[5, 5, 3]` would incorrectly report `5` as the second largest.

In [ ]:
# Brute force — sort, then walk down from the top for the first value
# that differs from the maximum. O(n log n) time.

def second_largest_brute(numbers):
    if len(numbers) < 2:
        raise ValueError("need at least two distinct values")
    sorted_desc = sorted(set(numbers), reverse=True)   # set() drops duplicates first
    if len(sorted_desc) < 2:
        raise ValueError("need at least two distinct values")
    return sorted_desc[1]

print(second_largest_brute([3, 1, 4, 1, 5, 9, 2, 6]))   # 6
print(second_largest_brute([5, 5, 3]))                   # 3

**Optimal:** The one-pass tracker from the pseudocode — O(n) time, O(1) space, and it doesn't need to look at any number twice or build a sorted copy of the whole list just to throw most of it away.

In [ ]:
# Optimal — one pass, tracking largest and second-largest together.
# O(n) time, O(1) space.

def second_largest_optimal(numbers):
    largest = second = float("-inf")
    for n in numbers:
        if n > largest:
            largest, second = n, largest      # old largest demoted to second
        elif second < n < largest:            # strictly between second and largest
            second = n
    if second == float("-inf"):
        raise ValueError("need at least two distinct values")
    return second

print(second_largest_optimal([3, 1, 4, 1, 5, 9, 2, 6]))   # 6
print(second_largest_optimal([5, 5, 3]))                   # 3

In [ ]:
# Tests

assert second_largest_optimal([3, 1, 4, 1, 5, 9, 2, 6]) == 6
assert second_largest_optimal([5, 5, 3]) == 3                 # edge: duplicate maximum
assert second_largest_optimal([9, 9, 5]) == 5                 # edge: duplicate maximum, unsorted
assert second_largest_optimal([-1, -2, -3]) == -2              # edge: all negative
assert second_largest_optimal([1, 2]) == 1                    # edge: exactly two elements

for bad_input in ([5], [5, 5, 5], []):
    try:
        second_largest_optimal(bad_input)
        assert False, f"expected ValueError for {bad_input}"
    except ValueError:
        pass   # edge: fewer than two distinct values correctly rejected

for numbers in ([3, 1, 4, 1, 5, 9, 2, 6], [5, 5, 3], [9, 9, 5], [-1, -2, -3]):
    assert second_largest_brute(numbers) == second_largest_optimal(numbers)

print("All Q13 tests passed.")

**Interview traps:**
- **Clarify "second largest" before coding — this is the whole question.** Does `[5, 5, 3]` return `5` (second position after sorting) or `3` (second *distinct* value)? Both are legitimate readings of an ambiguous prompt; the second is the far more common interview intent, but say so out loud rather than silently picking one and hoping it matches what's expected.
- **Initializing trackers to `0`, or to the list's first element, is a subtle bug generator.** `float("-inf")` (or `None`, checked explicitly) correctly handles all-negative lists (`[-1, -2, -3]`) and lists where the first element isn't special. Initializing to `0` breaks the moment every element in the list is negative.
- **The `elif second < n < largest` condition needs both bounds.** Checking only `n > second` (without `n < largest`) would let a *third* copy of the current maximum overwrite `second` incorrectly. Checking only `n < largest` (without `n > second`) would let a smaller, non-second-place number overwrite a legitimately larger `second`. Both comparisons are load-bearing.
- **`sorted(numbers, reverse=True)[1]` without deduplicating first** is the most common wrong brute-force answer — it silently returns a repeated maximum for inputs like `[5, 5, 3]` because it never removes duplicates before indexing. If you write this, you've implicitly picked the "second position" interpretation without saying so — which is a symptom of skipping restatement.
- **Fewer than two distinct values is a real edge case to handle deliberately** (raise, return `None`, or whatever you and the interviewer agree on) — don't let it fall through to an `IndexError` on an empty/short sorted list.

---
## Q14. Merge Sorted Lists

> **Merge Sorted Lists:** Given two sorted lists of integers, merge them into one sorted list.
> _Hint:_ Use two pointers to traverse both lists, appending the smaller element each time; alternatively, use Python's sorted on the concatenation (less optimal).

**Restate:** Given two lists, each already sorted ascending, produce a single sorted list containing every element from both (duplicates kept — this is a merge, not a union).

**Algorithmic thinking (no syntax):**
1. Point at the front of each list.
2. Compare the two elements currently being pointed at. Take the smaller one, place it onto the result, and move that list's pointer forward by one.
3. Repeat step 2 until you run off the end of one of the lists.
4. Whatever's left, unvisited, in the *other* list is already sorted and all bigger than everything placed so far — just append it wholesale.

The fact that both inputs are **already sorted** is the entire point of this question. If you ignore that fact and just concatenate + re-sort, you're throwing away information the problem handed you for free.

In [ ]:
# Brute force — ignore that the inputs are pre-sorted, concatenate and
# re-sort from scratch. O((n + m) log(n + m)) time.

def merge_sorted_brute(a, b):
    return sorted(a + b)

print(merge_sorted_brute([1, 3, 5], [2, 4, 6]))   # [1, 2, 3, 4, 5, 6]

**Optimal:** The two-pointer walk from the pseudocode. O(n + m) time — each element from each list is visited exactly once, and the fact that both are pre-sorted means we never need to compare against anything already placed.

In [ ]:
# Optimal — two pointers. O(n + m) time, O(n + m) space for the result.

def merge_sorted_optimal(a, b):
    result = []
    i, j = 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    result.extend(a[i:])   # leftover tail of whichever list still has elements
    result.extend(b[j:])   # at most one of these two extends does anything
    return result

print(merge_sorted_optimal([1, 3, 5], [2, 4, 6]))   # [1, 2, 3, 4, 5, 6]

In [ ]:
# Tests

assert merge_sorted_optimal([], []) == []                              # edge: both empty
assert merge_sorted_optimal([], [1, 2, 3]) == [1, 2, 3]                # edge: one empty
assert merge_sorted_optimal([1, 2, 3], []) == [1, 2, 3]                # edge: the other empty
assert merge_sorted_optimal([1, 3, 5], [2, 4, 6]) == [1, 2, 3, 4, 5, 6]
assert merge_sorted_optimal([1, 2, 3], [4, 5, 6]) == [1, 2, 3, 4, 5, 6]  # no interleaving needed
assert merge_sorted_optimal([1, 1, 2], [1, 3]) == [1, 1, 1, 2, 3]        # edge: duplicates across lists
assert merge_sorted_optimal([-3, -1], [-2, 0]) == [-3, -2, -1, 0]        # negative numbers

import random
random.seed(0)   # reproducible property-based check
for _ in range(20):
    a = sorted(random.randint(-20, 20) for _ in range(random.randint(0, 8)))
    b = sorted(random.randint(-20, 20) for _ in range(random.randint(0, 8)))
    assert merge_sorted_optimal(a, b) == merge_sorted_brute(a, b), (a, b)

print("All Q14 tests passed.")

**Interview traps:**
- **Forgetting the leftover tail is the single most common bug.** Once one list is exhausted, the `while` loop stops — if you forget the two `.extend()` calls (or equivalent), every remaining element in the longer list is silently dropped from the output. Always trace through a case where one list is much longer than the other.
- **`<=` vs `<` when comparing `a[i]` and `b[j]` controls stability with duplicates across the two lists.** Using `<=` (take from `a` on ties) means equal elements from `a` come before equal elements from `b` in the output — a defensible, common convention, but say which one you picked, since both are "correct" merges and only one matches what a stability-sensitive caller expects.
- **`sorted(a + b)` "works" but throws away the problem's core constraint.** If you offer this as your primary solution without immediately noting the two-pointer improvement, you've answered "merge two lists" instead of "merge two *sorted* lists," which is a different, easier problem.
- **`heapq.merge(a, b)` is the standard-library version of exactly this idea** — it's lazy (returns an iterator, doesn't materialize the full result unless you ask), and generalizes cleanly to merging *more than two* sorted iterables, which the two-pointer version doesn't do without real rework. Worth naming as the production answer, but implement the two-pointer version yourself to prove you understand why it works.
- **This is the exact merge step used inside merge sort** — if the interview continues toward "how would you sort a list from scratch," recognize that you already just wrote the hard half of it.

---
## Q15. Anagram Check

> **Anagram Check:** Check if two strings are anagrams of each other (contain exactly the same characters in a different order).
> _Hint:_ Sort both strings and compare, or count characters using a dictionary/Counter for each and compare frequencies.

**Restate:** Given two strings, decide whether one is a rearrangement of the other's exact characters — same characters, same counts of each, just possibly reordered. (We'll treat this case-sensitively and keep spaces, matching the case-sensitive convention set in Q3, unless told otherwise.)

**Algorithmic thinking (no syntax):**
1. If the two strings have different lengths, they can't possibly be anagrams — stop immediately.
2. Otherwise, tally up how many times each distinct character appears in the first string.
3. Do the same for the second string.
4. If the two tallies match exactly — same characters, same counts for each — they're anagrams.

Step 1 is a cheap, free early exit: a length mismatch settles the question in O(1) before you've done any real character-counting work at all.

In [ ]:
# Brute force — sort both strings into canonical order and compare.
# If two strings are anagrams, sorting both produces identical results
# by definition. O(n log n) time, dominated by the sort.

def is_anagram_brute(s1, s2):
    return sorted(s1) == sorted(s2)

print(is_anagram_brute("listen", "silent"))   # True
print(is_anagram_brute("hello", "world"))     # False

**Optimal:** Skip the sort entirely — count character frequencies with `Counter` and compare the counts directly. O(n) time instead of O(n log n), plus the free O(1) length check up front to short-circuit the obvious-no cases.

In [ ]:
from collections import Counter

# Optimal — length pre-check + Counter comparison. O(n) time.

def is_anagram_optimal(s1, s2):
    if len(s1) != len(s2):
        return False
    return Counter(s1) == Counter(s2)

print(is_anagram_optimal("listen", "silent"))   # True
print(is_anagram_optimal("hello", "world"))     # False

In [ ]:
# Tests

assert is_anagram_optimal("listen", "silent") is True
assert is_anagram_optimal("hello", "world") is False
assert is_anagram_optimal("", "") is True                   # edge: both empty
assert is_anagram_optimal("a", "a") is True                 # edge: single character, identical
assert is_anagram_optimal("a", "b") is False                # edge: single character, different
assert is_anagram_optimal("aabb", "abab") is True           # order fully scrambled
assert is_anagram_optimal("aabbcc", "aabbc") is False       # edge: different lengths -> False
assert is_anagram_optimal("Listen", "silent") is False      # case-sensitive, by convention here
assert is_anagram_optimal("aab", "abb") is False            # same length, same letters, different counts

for s1, s2 in [("listen", "silent"), ("hello", "world"), ("aabbcc", "aabbc"), ("", "")]:
    assert is_anagram_brute(s1, s2) == is_anagram_optimal(s1, s2)

print("All Q15 tests passed.")

**Interview traps:**
- **`"aab"` vs `"abb"` — same letters present, different counts — is not an anagram**, and it's the case that catches people who check "do both strings contain the same *set* of characters" (`set(s1) == set(s2)`) instead of the same *counts*. Sets throw away frequency information; anagram-ness depends entirely on frequency.
- **Skipping the length check isn't wrong, just slower to fail.** `Counter(s1) == Counter(s2)` alone already returns `False` for mismatched lengths — but naming the length pre-check explicitly shows you're thinking about cheap early exits, which matters more on the sorting approach where lengths differing wastes an entire O(n log n) sort before finding out.
- **Case and whitespace are judgment calls, exactly like Q3 and Q9** — the classic "is this word an anagram of that word" often wants case-insensitive comparison; the classic "are these two sentences anagrams" question often wants spaces stripped first. State the convention you're using before your code makes the decision for you silently.
- **`sorted(s1) == sorted(s2)` builds two full sorted *lists*, not strings** — comparing lists of characters, not strings — which still works correctly via `==` but is worth being precise about if asked what type each side of the comparison actually is.

---
## Q16. Sort by Second Element

> **Sort by Second Element:** Given a list of tuples, sort it by the second element of each tuple.
> _Hint:_ Use the `sorted()` function with a key, e.g., `sorted(list_of_tuples, key=lambda x: x[1])`.

**Restate:** Given a list of tuples (each with at least two elements), reorder the list so that the tuples appear in ascending order of their *second* element. Tuples that tie on the second element keep their original relative order.

**Algorithmic thinking (no syntax):**
1. Look at every pair of adjacent tuples in the list.
2. If a tuple's second element is bigger than the next tuple's second element, they're out of order — swap them.
3. Keep sweeping through and swapping until no more swaps are needed anywhere in the list.

That's a plain sorting algorithm (bubble sort), described generically — nothing about "second element" changes the *algorithm*, only what you compare. This question isn't really testing whether you can sort; it's testing whether you know Python already has a sort, and how to tell it what to compare by.

In [ ]:
# Brute force — hand-rolled bubble sort, comparing by index 1 explicitly.
# O(n^2) time. Nothing wrong with the *logic*; the problem is reinventing
# a sort that the standard library already provides, highly optimized.

def sort_by_second_brute(pairs):
    result = list(pairs)   # copy — don't mutate the caller's list
    n = len(result)
    for i in range(n):
        for j in range(n - 1 - i):
            if result[j][1] > result[j + 1][1]:
                result[j], result[j + 1] = result[j + 1], result[j]
    return result

print(sort_by_second_brute([(1, 3), (2, 1), (3, 2)]))   # [(2, 1), (3, 2), (1, 3)]

**Optimal:** Python's built-in `sorted()` is Timsort — O(n log n), implemented in C, and **stable** (equal elements keep their original relative order, which is exactly the tie-breaking behavior we want and specified in the restatement). All you supply is the *key*: a function that, given one tuple, returns the value to compare it by.

In [ ]:
from operator import itemgetter

# Optimal — built-in sort, keyed by index 1. O(n log n) time.

def sort_by_second_optimal(pairs):
    return sorted(pairs, key=lambda pair: pair[1])

# itemgetter(1) does the same thing as `lambda pair: pair[1]`, but skips the
# per-call overhead of invoking a Python-level function — a real (if usually
# small) speedup when sorting large lists.
def sort_by_second_itemgetter(pairs):
    return sorted(pairs, key=itemgetter(1))

print(sort_by_second_optimal([(1, 3), (2, 1), (3, 2)]))       # [(2, 1), (3, 2), (1, 3)]
print(sort_by_second_itemgetter([(1, 3), (2, 1), (3, 2)]))    # [(2, 1), (3, 2), (1, 3)]

In [ ]:
# Tests

assert sort_by_second_optimal([]) == []                                     # edge: empty list
assert sort_by_second_optimal([(1, 1)]) == [(1, 1)]                          # edge: single tuple
assert sort_by_second_optimal([(1, 3), (2, 1), (3, 2)]) == [(2, 1), (3, 2), (1, 3)]
assert sort_by_second_optimal([("x", 5), ("y", 5), ("z", 1)]) == [("z", 1), ("x", 5), ("y", 5)]  # stability: x before y, tie preserved
assert sort_by_second_optimal([(1, -3), (2, -1), (3, -2)]) == [(1, -3), (3, -2), (2, -1)]  # negative values

original = [(1, 3), (2, 1), (3, 2)]
sort_by_second_optimal(original)
assert original == [(1, 3), (2, 1), (3, 2)]   # sorted() doesn't mutate its input

for pairs in ([], [(1, 1)], [(1, 3), (2, 1), (3, 2)], [("x", 5), ("y", 5), ("z", 1)]):
    assert sort_by_second_brute(pairs) == sort_by_second_optimal(pairs) == sort_by_second_itemgetter(pairs)

print("All Q16 tests passed.")

**Interview traps:**
- **`sorted()` returns a new list; `list.sort()` mutates in place and returns `None`.** `pairs.sort(key=...)` is fine if you intend to mutate; `result = pairs.sort(key=...)` is a classic bug that leaves `result` as `None`. Know which one you're calling and why.
- **`key=`, not `cmp=`.** Older code (and other languages) uses a comparator function that takes two elements and returns negative/zero/positive. Python 3 removed `cmp` from `sorted()` entirely — you must express "what to compare" (`key`) rather than "how to compare two things" (`cmp`). If you write a comparator function out of habit, that's the tell you're thinking in an older paradigm.
- **Python's sort is stable — this is a guarantee, not an implementation detail**, and it's exactly why the "ties keep original order" part of the restatement is satisfiable at all with a single `sorted()` call. If you needed a *different* tie-breaking rule (say, by the first element as a secondary key), you'd make the key return a tuple: `key=lambda pair: (pair[1], pair[0])`.
- **Sorting descending is `reverse=True`, not a negated key** (usually) — `sorted(pairs, key=lambda p: p[1], reverse=True)`. Negating the key (`key=lambda p: -p[1]`) also works for numbers but breaks for non-numeric second elements (like strings) and is needlessly clever when `reverse=True` says the same thing more clearly.
- **Assuming every tuple has at least two elements.** If some tuples in the list are shorter, `pair[1]` raises `IndexError` mid-sort — worth naming as an input assumption you're relying on.

---
## Q17. Sum of Digits

> **Sum of Digits:** Calculate the sum of all digits of a given integer.
> _Hint:_ Use a loop with modulus/division to extract digits, or convert to string and sum the int-converted characters.

**Restate:** Given an integer `n`, add up its individual base-10 digits. We'll define this for negative numbers as "sum the digits of `|n|`" (the sign itself isn't a digit) — again, a convention worth stating rather than assuming.

**Algorithmic thinking (no syntax):**
1. Look at the last digit of the number.
2. Add it to a running total.
3. Chop that last digit off, leaving a shorter number.
4. Repeat until there are no digits left.

Like Q9's vowel-counting, this doesn't have a brute-force-vs-optimal complexity split — both approaches below visit each digit exactly once, O(d) for a `d`-digit number. The real choice is *arithmetic* (mod/divide, no string conversion) vs. *string-based* (convert to text, then process characters) — worth knowing both, since interviewers sometimes explicitly ban one or the other to see which you reach for.

In [ ]:
# "Brute force" — string-based: convert to text, sum each character back
# to an int. Simple and readable, at the cost of a string conversion.

def sum_of_digits_brute(n):
    return sum(int(digit) for digit in str(abs(n)))

print(sum_of_digits_brute(12345))   # 15
print(sum_of_digits_brute(-907))    # 16

**Optimal (arithmetic, no string conversion):** Same O(d) time, but works purely with integer arithmetic — the mod/divide loop from the pseudocode, directly. This is the version to offer if asked to avoid converting to a string, and it also directly demonstrates that you understand *why* `% 10` and `// 10` peel off digits (which the string version sidesteps entirely by outsourcing that to `str()`).

In [ ]:
# Optimal — arithmetic mod/divide loop. O(d) time, O(1) space,
# no string conversion.

def sum_of_digits_optimal(n):
    n = abs(n)
    total = 0
    while n > 0:
        total += n % 10   # peel off the last digit
        n //= 10           # drop it
    return total

print(sum_of_digits_optimal(12345))   # 15
print(sum_of_digits_optimal(-907))    # 16

In [ ]:
# Tests

assert sum_of_digits_optimal(0) == 0                     # edge: zero
assert sum_of_digits_optimal(5) == 5                     # edge: single digit
assert sum_of_digits_optimal(12345) == 15
assert sum_of_digits_optimal(-907) == 16                 # edge: negative — sign not counted
assert sum_of_digits_optimal(1000000) == 1               # edge: lots of zero digits
assert sum_of_digits_optimal(999) == 27                  # edge: all same digit, carries no meaning here

for n in list(range(-1000, 1000, 7)):
    assert sum_of_digits_brute(n) == sum_of_digits_optimal(n), n

print("All Q17 tests passed.")

**Interview traps:**
- **Negative numbers break the mod/divide approach if you don't take `abs()` first.** In Python, `-907 % 10` is `3`, not `-3` (Python's `%` always returns a result with the same sign as the divisor) — so the raw mod/divide loop on a negative number doesn't crash, it silently computes the *wrong* answer with subtly wrong-signed digits. Always normalize the sign before the loop, and know *why*: this is a genuine Python-specific gotcha (C/Java's `%` behaves differently on negatives).
- **The string approach needs `abs()` too, for a different reason** — `str(-907)` is `"-907"`, and `int('-')` raises `ValueError`. Both approaches need the sign handled explicitly; neither gets it "for free."
- **`n == 0` must produce `0`, not skip the loop and return something uninitialized.** The `while n > 0` loop correctly never executes for `n = 0`, but only because `total` was initialized to `0` before the loop — trace this edge case explicitly rather than assuming it "just works."
- **This is the building block for digit-based follow-ups** (digital root — repeatedly sum digits until one digit remains; Q18's digit reversal uses the identical mod/divide peeling step). Recognizing the shared pattern across questions is worth pointing out unprompted.

---
## Q18. Reverse Integer Digits

> **Reverse Integer Digits:** Reverse the digits of an integer. E.g., 123 -> 321 (assuming no overflow issues).
> _Hint:_ Use modulus and division repeatedly to construct the reversed number, or convert to string and reverse that (mind the sign for negative numbers).

**Restate:** Given an integer, produce the integer formed by reversing the order of its digits, keeping the original sign (so `-123` reverses to `-321`, not `321`).

**Algorithmic thinking (no syntax):**
1. Remember the sign of the original number, then set it aside — work with the positive version.
2. Start a result at zero.
3. Look at the last digit of the remaining number.
4. Shift the result one place to the left (multiply by ten) and add that digit into the newly opened last position.
5. Chop the digit off the remaining number (same as Q17's peeling step) and repeat from step 3 until nothing's left.
6. Re-apply the sign you set aside in step 1.

Step 4 is the one genuinely new idea here versus Q17: you're not just *accumulating* a sum, you're *constructing a new number digit by digit*, and each newly-peeled digit needs to land in the correct place value.

In [ ]:
# "Brute force" — string-based: strip the sign, reverse the text,
# convert back to int, then reapply the sign.

def reverse_integer_brute(n):
    sign = -1 if n < 0 else 1
    reversed_str = str(abs(n))[::-1]
    return sign * int(reversed_str)

print(reverse_integer_brute(123))    # 321
print(reverse_integer_brute(-123))   # -321
print(reverse_integer_brute(120))    # 21 — leading zero disappears, see traps below

**Optimal (arithmetic, no string conversion):** The mod/divide construction from the pseudocode. Both approaches are O(d) time — there's no complexity win here, same as Q17 — but this is the version to offer when explicitly asked to avoid strings, and it's the one the classic LeetCode framing of this question ("reverse digits *without* overflowing a 32-bit int") is actually built around, since you can check for overflow *before* each multiply instead of after building a possibly-oversized string.

In [ ]:
# Optimal — arithmetic construction. O(d) time, O(1) space.

def reverse_integer_optimal(n):
    sign = -1 if n < 0 else 1
    n = abs(n)
    result = 0
    while n > 0:
        digit = n % 10
        result = result * 10 + digit   # shift left, drop the new digit into place
        n //= 10
    return sign * result

print(reverse_integer_optimal(123))    # 321
print(reverse_integer_optimal(-123))   # -321
print(reverse_integer_optimal(120))    # 21

In [ ]:
# Tests

assert reverse_integer_optimal(0) == 0                    # edge: zero
assert reverse_integer_optimal(5) == 5                     # edge: single digit
assert reverse_integer_optimal(123) == 321
assert reverse_integer_optimal(-123) == -321               # edge: negative, sign preserved
assert reverse_integer_optimal(120) == 21                  # edge: trailing zero disappears
assert reverse_integer_optimal(100) == 1                   # edge: mostly zeros
assert reverse_integer_optimal(1000000003) == 3000000001   # large number, Python has no overflow

for n in list(range(-10000, 10000, 37)):
    assert reverse_integer_brute(n) == reverse_integer_optimal(n), n

print("All Q18 tests passed.")

**Interview traps:**
- **Trailing zeros vanishing is correct, not a bug — but say so.** `120` reversed is mathematically `021`, and `021` as an integer is just `21` — a leading zero in a number has no value. If asked "what should `reverse(120)` be," confirming `21` (not `021`, which isn't representable as a distinct integer) shows you understand *why*, rather than being surprised by your own code's output.
- **Sign handling is the same two-step as Q17**: strip it before processing, reapply it after. Forgetting this on the arithmetic version doesn't crash — Python's `%` on negative numbers (see Q17's trap) just quietly produces the wrong digits.
- **The "no overflow" caveat in the prompt is a direct pointer to the classic constrained version of this question** (LeetCode 7), which asks you to return `0` if the reversed number would overflow a 32-bit signed integer (`-2^31` to `2^31 - 1`). Python integers don't have that ceiling, so this specific failure mode literally cannot happen here — but naming the constraint and explaining why Python sidesteps it (arbitrary-precision `int`, same point made about `factorial` in Q4) is exactly the kind of language-aware answer that stands out.
- **`abs()` on the theoretical minimum 32-bit integer (`-2**31`) actually overflows in fixed-width languages** (there's no positive counterpart in two's-complement 32-bit range) — a sharp follow-up question that doesn't apply to Python's arbitrary-precision ints, but is worth having heard of if the interviewer is testing general systems knowledge rather than Python specifically.

---
## Q19. All Unique Characters

> **All Unique Characters:** Determine if a string has all unique characters (no duplicates).
> _Hint:_ Use a set to track seen characters or compare length of set of characters to length of string.

**Restate:** Given a string, decide whether every character in it appears exactly once (case-sensitive, same convention as Q3/Q15 unless told otherwise).

**Algorithmic thinking (no syntax):**
1. Keep a "memory" of every character you've seen so far, starting empty.
2. Walk the string one character at a time.
3. If the current character is already in your memory, you've found a duplicate — stop immediately, the answer is no.
4. Otherwise, add it to your memory and keep going.
5. If you reach the end without ever finding a duplicate, the answer is yes.

This is structurally identical to Q12 (remove duplicates) — same "have I seen this before?" memory-check pattern — except here we only care about the yes/no answer, and step 3 gives us an early exit the moment the answer is settled.

In [ ]:
# Brute force — compare every pair of characters directly. O(n^2) time.

def all_unique_brute(s):
    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            if s[i] == s[j]:
                return False
    return True

print(all_unique_brute("abcdef"))   # True
print(all_unique_brute("hello"))    # False — repeated 'l'

**Optimal:** A `set` tracking seen characters, with early exit — O(n) time. There's also a neat O(1)-time *pre-check* worth knowing: if the string is longer than the number of distinct characters possible in its alphabet (128 for ASCII, for instance), it's guaranteed to have a duplicate by the pigeonhole principle, without looking at a single character.

In [ ]:
# Optimal — set with early exit. O(n) time, O(min(n, alphabet size)) space.

def all_unique_optimal(s):
    if len(s) > 128:              # pigeonhole: can't be unique if longer than the ASCII alphabet
        return False
    seen = set()
    for char in s:
        if char in seen:
            return False           # early exit — no need to look at the rest
        seen.add(char)
    return True

print(all_unique_optimal("abcdef"))   # True
print(all_unique_optimal("hello"))    # False

# Equivalent one-liner: compares total length to the count of distinct
# characters. Correct, but always builds the full set -- no early exit,
# so it can't stop the instant a duplicate is found the way the loop can.
def all_unique_length_compare(s):
    return len(set(s)) == len(s)

print(all_unique_length_compare("abcdef"))   # True

In [ ]:
# Tests

assert all_unique_optimal("") is True                       # edge: empty string — vacuously unique
assert all_unique_optimal("a") is True                       # edge: single character
assert all_unique_optimal("aa") is False                     # edge: immediate duplicate
assert all_unique_optimal("abcdef") is True
assert all_unique_optimal("hello") is False
assert all_unique_optimal("Aa") is True                       # case-sensitive: 'A' and 'a' differ
assert all_unique_optimal("x" * 129) is False                 # edge: pigeonhole pre-check triggers

for s in ["", "a", "aa", "abcdef", "hello", "Aa", "abcdefg"]:
    assert all_unique_brute(s) == all_unique_optimal(s) == all_unique_length_compare(s), s

print("All Q19 tests passed.")

**Interview traps:**
- **`len(set(s)) == len(s)` always scans the whole string, even if character 2 and character 3 are an obvious duplicate.** It's a perfectly correct one-liner, but if you're asked to add an early exit for large inputs, you need the explicit loop version — know both and know which one the question is actually asking for.
- **The pigeonhole pre-check assumes a bounded alphabet.** It's correct for ASCII (128) or extended ASCII (256), but breaks the moment the input can contain arbitrary Unicode, which has far more than 128 distinct code points — don't apply this shortcut without confirming what character set is in play.
- **Case sensitivity, again** — `"Aa"` has all-unique characters *if* you're treating case-sensitively (the default assumption here); if the question actually means "unique letters regardless of case," you'd lowercase first, exactly as in Q3/Q9/Q15. Every one of these string questions hinges on the same unstated decision — noticing the pattern across all of them is worth saying out loud.
- **Don't reach for `sorted()` + adjacent-comparison as your *primary* answer** (sort the characters, then check if any neighbor matches) — it works and is O(n log n), a legitimate middle ground between brute force and the set-based O(n), but leading with it instead of the set-based version undersells that you know the faster option exists.

---
## Q20. List Intersection

> **List Intersection:** Find the common elements (intersection) of two lists.
> _Hint:_ Use a set for one list and iterate through the other, checking membership (for efficiency).

**Restate:** Given two lists, produce the elements that appear in *both*. We'll return each common element once (a true set intersection, not counting how many times it repeats in either input), in the order it first appears in the first list — the same "preserve order" convention as Q12, stated explicitly since the prompt doesn't.

**Algorithmic thinking (no syntax):**
1. Walk through the first list, one element at a time.
2. For each element, ask: "does this also show up somewhere in the second list?"
3. If yes — and you haven't already included it — add it to the result.
4. Once you've walked the whole first list, the result is the answer.

Step 2 is where all the cost lives. "Does this show up in the second list?" is itself a search — and how you implement *that* search is the entire brute-force-vs-optimal story for this question, identical in shape to the membership-check upgrade in Q12 and Q19.

In [ ]:
# Brute force — membership check against list2 is itself a linear scan.
# O(n * m) time for lists of length n and m.

def list_intersection_brute(list1, list2):
    result = []
    for item in list1:
        if item in list2 and item not in result:
            result.append(item)
    return result

print(list_intersection_brute([1, 2, 3, 4], [3, 4, 5, 6]))   # [3, 4]

**Optimal:** Convert the second list to a `set` once, up front — O(1) average membership checks instead of O(m) — and track "already added" with a `set` too, instead of the brute force's `item not in result` (which was its own hidden O(k) linear scan). This takes the whole thing to O(n + m) time.

In [ ]:
# Optimal — set for the lookup side, set for the dedup side.
# O(n + m) time, O(n + m) space.

def list_intersection_optimal(list1, list2):
    lookup = set(list2)
    seen = set()
    result = []
    for item in list1:
        if item in lookup and item not in seen:
            result.append(item)
            seen.add(item)
    return result

print(list_intersection_optimal([1, 2, 3, 4], [3, 4, 5, 6]))   # [3, 4]

# If order doesn't matter, Python's own set intersection operator is the
# most direct expression of the idea -- but it returns a set, so any
# order from either input list is lost.
def list_intersection_set_only(list1, list2):
    return set(list1) & set(list2)

print(list_intersection_set_only([1, 2, 3, 4], [3, 4, 5, 6]))   # {3, 4} — order not guaranteed

In [ ]:
# Tests

assert list_intersection_optimal([], []) == []                          # edge: both empty
assert list_intersection_optimal([1, 2, 3], []) == []                    # edge: one empty
assert list_intersection_optimal([], [1, 2, 3]) == []                    # edge: the other empty
assert list_intersection_optimal([1, 2, 3], [4, 5, 6]) == []             # edge: no overlap at all
assert list_intersection_optimal([1, 2, 3], [1, 2, 3]) == [1, 2, 3]      # edge: fully identical
assert list_intersection_optimal([1, 2, 3, 4], [3, 4, 5, 6]) == [3, 4]
assert list_intersection_optimal([1, 1, 2, 2], [1, 2]) == [1, 2]         # edge: duplicates in list1 collapse
assert list_intersection_optimal([3, 1, 2], [2, 3]) == [3, 2]            # order follows list1, not list2

for list1, list2 in [([1, 2, 3, 4], [3, 4, 5, 6]), ([1, 1, 2, 2], [1, 2]), ([], [1, 2])]:
    assert (
        sorted(list_intersection_brute(list1, list2))
        == sorted(list_intersection_optimal(list1, list2))
        == sorted(list_intersection_set_only(list1, list2))
    )

print("All Q20 tests passed.")

**Interview traps:**
- **`set(list1) & set(list2)` is the fastest and most direct answer if order and duplicate-count don't matter** — say so up front. It's O(n + m) time, same as the hand-rolled version, just built on the language's native set operation instead of a manual loop. Only reach for the manual loop if you need to preserve order from one of the inputs, which a `set` fundamentally cannot do.
- **"Intersection" is ambiguous about duplicates, exactly like Q10's word counting was ambiguous about case.** Does `[1, 1, 2]` ∩ `[1, 2]` return `[1, 2]` (each common *value*, once) or `[1, 1, 2]` (matching each occurrence in `list1` that has a corresponding occurrence in `list2`, i.e. a multiset intersection)? We picked the former here; state your choice, since both are "the intersection" under different definitions.
- **Elements must be hashable to go in a `set`**, exactly the same constraint as Q12's duplicate removal — a list of unhashable items (like nested lists) forces you back to the O(n·m) brute-force membership check, and you should recognize that limitation rather than assuming `set()` always applies.
- **This question is the mirror image of "subset check"** (is every element of one list contained in another?) — the same "convert one side to a `set`, then check membership" upgrade applies there too. If a follow-up asks for subset instead of intersection, you already have the right tool in hand.